# Tree Segmentation & Height Extraction from CHM

Detects individual trees from canopy height models (CHMs), clips results to GEDI footprints, and exports per-tree height and location data for downstream biomass analysis.

**Resolutions processed:** 1.0m, 0.5m, 0.25m, 0.1m

**Outputs** (written to `../outputs/`):
- `treetops_<res>.gpkg` — treetop locations with height
- `tree_heights_<res>.csv` — treeID, x, y, height_m

## 1. Install & Load Packages

In [1]:
# Uncomment if packages are not yet installe
# install.packages(c('terra', 'sf', 'lidR', 'rlas', 'dplyr'), repos = 'https://cran.r-project.org')

library(terra)
library(sf)
library(lidR)
library(dplyr)

terra 1.7.23

Linking to GEOS 3.10.2, GDAL 3.4.1, PROJ 7.2.1; sf_use_s2() is TRUE


Attaching package: 'lidR'


The following object is masked from 'package:sf':

    st_concave_hull



Attaching package: 'dplyr'


The following objects are masked from 'package:terra':

    intersect, union


The following objects are masked from 'package:stats':

    filter, lag


The following objects are masked from 'package:base':

    intersect, setdiff, setequal, union




## 2. Configuration

Edit this cell to change resolutions, smoothing kernels, or window size (ws) algorithms.
- `label` — human-readable resolution name
- `suffix` — used in output filenames
- `kernel` — smoothing kernel size (matrix of 1s)
- `ws` — local maxima filter window size; can be a fixed number or a function of height

In [2]:
chm_configs <- list(
  list(
    label  = "1.0m",
    suffix = "1m",
    kernel = matrix(1, 3, 3),
    ws     = 3.0
  ),
  list(
    label  = "0.5m",
    suffix = "05m",
    kernel = matrix(1, 3, 3),
    ws     = 3.5
  ),
  list(
    label  = "0.25m",
    suffix = "025m",
    kernel =  matrix(1, 3, 3),       #kernel = matrix(1, 5, 5),
    ws     = 4                 #ws     = function(x) pmin(pmax(2.0 + 0.1 * x, 2.0), 6.0)
  ),
  list(
    label  = "0.1m",
    suffix = "01m",
    kernel =  matrix(1, 3, 3),       #kernel = matrix(1, 5, 5),
    ws     = 5                    #ws     = function(x) pmin(pmax(2.0 + 0.1 * x, 2.0), 6.0)
)
)

## 3. Set Up Output Directory

In [ ]:
output_dir <- "C:/Users/kdavis99/OneDrive - Cal Poly/Tree Biomass Estimation Research - Documents/GitHub_Repository/Biomass_Monitoring_Research/outputs/Bartleson Tree Segmentation"
if (!dir.exists(output_dir)) dir.create(output_dir, recursive = TRUE)
cat("Output directory:", output_dir, "\n")

## 4. Load GEDI Footprints

Footprints are shared across all CHM resolutions. A file chooser dialog will open.

In [8]:
message("Select your GEDI footprints shapefile (.shp or .gpkg).")
footprints <- st_read(file.choose())
cat("Loaded", nrow(footprints), "footprints\n")
print(st_crs(footprints))

Select your GEDI footprints shapefile (.shp or .gpkg).



Warning message in CPL_read_ogr(dsn, layer, query, as.character(options), quiet, :
"GDAL Message 1: This version of GeoPackage user_version=0x000028A0 (10400, v1.4.0) on 'C:\Users\kdavis99\OneDrive - Cal Poly\Tree Biomass Estimation Research - Documents\GitHub_Repository\selected_footprints.gpkg' may only be partially supported"


Reading layer `FindExistingLocationsOutput' from data source 
  `C:\Users\kdavis99\OneDrive - Cal Poly\Tree Biomass Estimation Research - Documents\GitHub_Repository\selected_footprints.gpkg' 
  using driver `GPKG'
Simple feature collection with 24 features and 31 fields
Geometry type: MULTIPOLYGON
Dimension:     XY
Bounding box:  xmin: -13419370 ymin: 4175280 xmax: -13417750 ymax: 4176014
Projected CRS: WGS 84 / Pseudo-Mercator
Loaded 24 footprints
Coordinate Reference System:
  User input: WGS 84 / Pseudo-Mercator 
  wkt:
PROJCRS["WGS 84 / Pseudo-Mercator",
    BASEGEOGCRS["WGS 84",
        DATUM["World Geodetic System 1984",
            ELLIPSOID["WGS 84",6378137,298.257223563,
                LENGTHUNIT["metre",1]]],
        PRIMEM["Greenwich",0,
            ANGLEUNIT["degree",0.0174532925199433]],
        ID["EPSG",4326]],
    CONVERSION["Popular Visualisation Pseudo-Mercator",
        METHOD["Popular Visualisation Pseudo Mercator",
            ID["EPSG",1024]],
        PARAMETER[

## 5. Helper Functions

In [12]:
# Reproject source to match target CRS if needed
align_crs <- function(source, target) {
  if (st_crs(source) != st_crs(target)) {
    message("CRS mismatch - reprojecting to match CHM...")
    return(st_transform(source, st_crs(target)))
  }
  message("CRS match confirmed.")
  return(source)
}

# Process a single CHM config: smooth, detect trees, clip to footprints, export
process_chm <- function(cfg, footprints, output_dir, chm_path = NULL) {
  cat("\n==================================================\n")
  cat("Processing", cfg$label, "CHM\n")
  cat("==================================================\n")

  # Load CHM — use provided path or fall back to file chooser
  if (is.null(chm_path)) {
    message("Select your ", cfg$label, " CHM raster file.")
    chm_path <- file.choose()
  }
  chm <- rast(chm_path)
  cat("Loaded CHM:", chm_path, "\n")

  # Smooth CHM
  chm_smooth <- terra::focal(chm, w = cfg$kernel, fun = median, na.rm = TRUE)
  plot(chm_smooth, col = height.colors(50),
       main = paste(cfg$label, "- Smoothed CHM"))

  # Detect treetops
  ttops <- locate_trees(las = chm_smooth, algorithm = lmf(ws = cfg$ws))
  ttops$height <- st_coordinates(ttops)[, "Z"]
  cat("Detected", nrow(ttops), "trees\n")

  # Align footprints CRS and clip trees to footprints
  footprints_aligned <- align_crs(footprints, ttops)
  ttops_in_footprints <- st_intersection(ttops, footprints_aligned)
  cat(nrow(ttops_in_footprints), "trees fall within footprints\n")

  # Tree counts per footprint
  tree_counts <- ttops_in_footprints |>
    st_drop_geometry() |>
    group_by(reading_order_id) |>
    summarise(tree_count = n(), .groups = "drop")
  print(tree_counts)

  # Export GeoPackage
  gpkg_out <- file.path(output_dir, paste0("treetops_", cfg$suffix, ".gpkg"))
  st_write(st_zm(ttops, drop = TRUE), gpkg_out, delete_dsn = TRUE)
  cat("Saved:", gpkg_out, "\n")

  # Export CSV (includes height_m for every tree)
  tree_data <- data.frame(
    treeID   = ttops$treeID,
    x        = st_coordinates(ttops)[, "X"],
    y        = st_coordinates(ttops)[, "Y"],
    height_m = ttops$height
  )
  csv_out <- file.path(output_dir, paste0("tree_heights_", cfg$suffix, ".csv"))
  write.csv(tree_data, csv_out, row.names = FALSE)
  cat("Saved:", csv_out, "\n")

  # Return everything needed for later visualization and summary
  return(list(
    tree_data           = tree_data,
    chm                 = chm,
    ttops               = ttops,
    footprints_aligned  = footprints_aligned,
    ttops_in_footprints = ttops_in_footprints
  ))
}

## 6. Run Processing

This will prompt you to select a CHM file for **each** resolution in order:
1. 1.0m CHM
2. 0.5m CHM
3. 0.25m CHM
4. 0.1m CHM

In [ ]:
raster_dir <- "C:/Users/kdavis99/OneDrive - Cal Poly/Tree Biomass Estimation Research - Documents/GitHub_Repository/Rasters"

chm_paths <- list(
  "1m"   = file.path(raster_dir, "bart_chm_1.0m_clipped.tif"),
  "05m"  = file.path(raster_dir, "bart_chm_0.5m_clipped.tif"),
  "025m" = file.path(raster_dir, "bart_chm_0.25m_clipped.tif"),
  "01m"  = file.path(raster_dir, "bart_chm_0.1m_clipped.tif")
)

results <- mapply(process_chm, chm_configs,
                  MoreArgs = list(footprints = footprints, output_dir = output_dir),
                  chm_path = chm_paths,
                  SIMPLIFY = FALSE)

names(results) <- sapply(chm_configs, `[[`, "suffix")

## 7. Height Summary

Prints per-tree height data and summary statistics (min, max, mean) for each resolution.

In [ ]:
for (i in seq_along(results)) {
  label <- chm_configs[[i]]$label
  td    <- results[[i]]$tree_data

  cat(sprintf("\n--- %s ---\n", label))
  cat(sprintf("  Trees detected : %d\n", nrow(td)))
  cat(sprintf("  Height min     : %.2f m\n", min(td$height_m, na.rm = TRUE)))
  cat(sprintf("  Height max     : %.2f m\n", max(td$height_m, na.rm = TRUE)))
  cat(sprintf("  Height mean    : %.2f m\n", mean(td$height_m, na.rm = TRUE)))
  cat(sprintf("  Height median  : %.2f m\n", median(td$height_m, na.rm = TRUE)))
  cat("\n  First 10 trees:\n")
  print(head(td, 10))
}

## 8. Detection Summary Table

In [ ]:
summary_df <- data.frame(
  resolution          = sapply(chm_configs, `[[`, "label"),
  trees_total         = sapply(results, function(r) nrow(r$tree_data)),
  trees_in_footprints = sapply(results, function(r) nrow(r$ttops_in_footprints)),
  mean_height_m       = sapply(results, function(r) round(mean(r$tree_data$height_m, na.rm = TRUE), 2)),
  max_height_m        = sapply(results, function(r) round(max(r$tree_data$height_m, na.rm = TRUE), 2))
)

cat("\n=== Detection Summary ===\n")
print(summary_df)

## 9. Visualization - CHM with Detected Treetops

For each resolution, plots the full CHM with:
- **Blue crosses** - all detected treetops (sized by height)
- **Red crosses** - treetops within GEDI footprints
- **Black outlines** - GEDI footprint boundaries

In [ ]:
for (i in seq_along(results)) {
  r      <- results[[i]]
  label  <- chm_configs[[i]]$label
  n_all  <- nrow(r$tree_data)
  n_fp   <- nrow(r$ttops_in_footprints)

  # Scale point size by height
  heights    <- r$tree_data$height_m
  cex_scaled <- pmin(pmax(heights / max(heights, na.rm = TRUE) * 1.5, 0.3), 1.5)

  # Plot CHM
  plot(
    r$chm,
    col  = height.colors(50),
    main = sprintf("%s CHM - %d trees total, %d in footprints", label, n_all, n_fp),
    axes = TRUE
  )

  # All treetops (blue)
  points(
    r$tree_data$x, r$tree_data$y,
    pch = 3, col = adjustcolor("steelblue", alpha.f = 0.6),
    cex = cex_scaled
  )

  # GEDI footprint boundaries
  plot(st_geometry(r$footprints_aligned), border = "black", lwd = 1.5, add = TRUE)

  # Treetops inside footprints (red)
  fp_coords <- st_coordinates(r$ttops_in_footprints)
  points(
    fp_coords[, "X"], fp_coords[, "Y"],
    pch = 3, col = "red", cex = 0.8, lwd = 1.2
  )

  # Legend
  legend(
    "topright",
    legend = c("All trees", "In footprint", "Footprint boundary"),
    col    = c("steelblue", "red", "black"),
    pch    = c(3, 3, NA),
    lty    = c(NA, NA, 1),
    lwd    = c(NA, NA, 1.5),
    bg     = "white",
    cex    = 0.8
  )
}